# Lab 4 demo — MycoVault MV-200 multi-agent helpdesk

Walkthrough of Parts **A–E** with local Ollama (`llama3.1:8b` + `nomic-embed-text`).

- Live demos for A–D (small samples).
- Part E loads tables already computed in `results/` (no full re-eval).
- After each part: markdown answers taken from `REPORT.md`.

### Architecture (report §1)

Customer → **Coordinator** (blackboard) → Triage → Retriever → KB → Specialist → Verifier → Escalation (+ ticket tools).
Empty retrieval or `NOT_IN_MANUAL` never invents an answer. Embeddings are cached under `.cache/` keyed by the manual’s SHA-256 (Task B2).

## Setup

Needs Ollama running and models pulled (`llama3.1:8b`, `nomic-embed-text`). See `README.md`.

In [3]:
import sys
print(sys.executable)

c:\Python311\python.exe


In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "kb.py").exists():
    # If the kernel CWD is elsewhere, jump to this notebook's folder
    ROOT = Path(r"E:\MasterSem1\An2Sem1\MultiAgentic AI\lab4_helpdesk")
sys.path.insert(0, str(ROOT))

import ollama
import config
from baseline import ask_baseline, ask_rag
from kb import KnowledgeBase
from coordinator import Coordinator
from IPython.display import Markdown, display

listed = ollama.list()
models = listed.models if hasattr(listed, "models") else listed.get("models", [])
names = []
for m in models:
    if hasattr(m, "model"):
        names.append(m.model)
    elif hasattr(m, "name"):
        names.append(m.name)
    elif isinstance(m, dict):
        names.append(m.get("model") or m.get("name") or "")

need = ["llama3.1:8b", "nomic-embed-text"]
missing = [n for n in need if not any(str(x).startswith(n.split(":")[0]) for x in names)]
if missing:
    raise RuntimeError(f"Missing models {missing}. Run: ollama pull ...\nInstalled: {sorted(names)}")

print("Ready.")
print("Chat model:", config.CHAT_MODEL)
print("Embed model:", config.EMBED_MODEL)
print("Project:", ROOT)

Ready.
Chat model: llama3.1:8b
Embed model: nomic-embed-text
Project: e:\MasterSem1\An2Sem1\MultiAgentic AI\lab4_helpdesk


## Part A — Baseline (ungrounded)

Generic support prompt, **no manual**. Sample of the Appendix B questions (full Q1–10 in `results/partA_baseline.json`).

In [5]:
sample_a = [
    "My display shows E07. What should I do?",
    "What humidity for Lion's Mane?",
    "Can I clean the inside with bleach?",
    "What does E12 mean?",
]

for q in sample_a:
    print("=" * 60)
    print("Q:", q)
    print(ask_baseline(q))
    print()

Q: My display shows E07. What should I do?
E07 is a specific error code for the MycoVault MV-200. It indicates a problem with the temperature control system. 

To troubleshoot, I'd like to guide you through a few steps. 

First, please check that the cabinet is properly plugged into a power outlet and that the outlet is working. Next, verify that the temperature settings are correct and that the cabinet is set to the desired temperature range for your specific mushroom species.

If the issue persists, I recommend checking the temperature probe and ensuring it's securely seated in the correct location. You can also try resetting the cabinet by unplugging it, waiting for 30 seconds, and then plugging it back in.

If none of these steps resolve the issue, I'd be happy to assist you further or schedule a remote support session to investigate the problem in more detail.

Q: What humidity for Lion's Mane?
Lion's Mane (Hericium erinaceus) is a wonderful species to cultivate. For optimal growt

### Part A — answers (from REPORT.md §3 + §8)

**1. How many correct / confidently wrong?**  
~0 fully correct on Q1–10. A couple are only vaguely right (bleach / tap water discouraged) with wrong details; ~8–10 are confidently wrong (invented numbers and procedures).

**2. Did the model say “I don’t know”? Why rare?**  
No. Helpful-completion training + a support-agent prompt → fluent guessing beats refusing.

**3. Dangerous wrong answers?**  
- **Safety:** wrong E07/E12 (misses Quarantine / FFP2); bleach advice that skips CO₂-sensor damage.  
- **Warranty / money:** “5-year warranty”; vinegar descaling; wrong humidity / filter intervals.

**Examples from the saved full run:** Q9 invents a vinegar soak (manual: 5 % citric + CLEAN); Q10 invents “E12 LED candelabra base” (manual: SporeEye contamination / Quarantine).

## Part B — Knowledge base + single RAG agent

Chunk → embed (`nomic-embed-text`) → cosine search. Cache under `.cache/` (SHA-256 of the manual).

In [6]:
kb = KnowledgeBase()
print(f"{len(kb.chunks)} chunks | chunk_size={kb.chunk_size} | sha256={kb.sha256[:12]}...")

print("\n--- Task B1: top-3 ---")
for query in ["E07", "lion's mane humidity", "bleach"]:
    print(f"\nQuery: {query}")
    for hit in kb.search(query, k=3):
        print(f"  {hit['score']:.3f}  [{hit['section']}]  chunk {hit['id']}")

11 chunks | chunk_size=900 | sha256=1c59b30d4a5e...

--- Task B1: top-3 ---

Query: E07
  0.627  [7. Error Codes]  chunk 7
  0.540  [7. Error Codes]  chunk 8
  0.526  [9. Warranty and Service]  chunk 10

Query: lion's mane humidity
  0.666  [5. Cultivation Profiles (Fruiting stage)]  chunk 5
  0.570  [2. Technical Specifications]  chunk 2
  0.545  [1. Product Overview]  chunk 1

Query: bleach
  0.498  [5. Cultivation Profiles (Fruiting stage)]  chunk 5
  0.492  [9. Warranty and Service]  chunk 10
  0.481  [8. Safety]  chunk 9


In [7]:
print("--- Task B3: ask_rag on the same samples ---\n")
for q in sample_a:
    print("=" * 60)
    print("Q:", q)
    print(ask_rag(q))
    print()

--- Task B3: ask_rag on the same samples ---

Q: My display shows E07. What should I do?
[Section: 7. Error Codes]
Move the unit to fresh air (or open a window), empty the chamber, then hold FAN + LIGHT for 5 s. Recalibration takes 8 minutes.

Q: What humidity for Lion's Mane?
[Section: 5. Cultivation Profiles (Fruiting stage)]
Lion's Mane | 18–21 | 94 | 700 | 8

Q: Can I clean the inside with bleach?
[Section: 6. Maintenance] Never use chlorine bleach: chlorine vapour permanently damages the NDIR CO2 sensor and voids the warranty.

Q: What does E12 mean?
[Section: 7. Error Codes]
E12 | Contamination detected by SporeEye | The unit enters Quarantine Mode (FAE stops, door lock on). Wear an FFP2 mask, remove and seal the affected block in a bag, disinfect the chamber, then press OK to exit Quarantine.



### Part B — answers (from REPORT.md §3)

**B1 retrieval:** E07 and Lion’s Mane hit the right sections; **“bleach”** often ranks Cultivation / Warranty / Safety above Maintenance — dense retrieval is weak on rare keywords.

**B3:** On the full Q1–10 run, single RAG grounded **9/10**; Q6 (5 GHz-only router) once returned `NOT_IN_MANUAL` (missed the 2.4 GHz fact).

**τ = 0.50** (from `results/tau_sweep.md`): same hit@5 as 0.45 with fewer chunks. Out-of-manual Q18 still scores ~0.75 at top, so **refusal cannot rely on empty retrieval alone** — Specialist / `NOT_IN_MANUAL` must decline.

## Part C — Multi-agent coordinator

Triage → Retriever → Specialist → Verifier → (maybe) Escalation. Print the blackboard like `--trace`.

In [8]:
coord = Coordinator(KnowledgeBase())

q1 = "My display shows E07. What should I do?"
reply1 = coord.answer(q1)
print("Q:", q1)
print(coord.trace())
print("\nReply:", reply1)

history = [
    {"role": "user", "content": q1},
    {"role": "assistant", "content": reply1},
]
q2 = "and how long does that take?"
reply2 = coord.answer(q2, history)
print("\n" + "=" * 60)
print("Follow-up:", q2)
print(coord.trace())
print("\nReply:", reply2)

Q: My display shows E07. What should I do?
[triage -> coordinator] inform
    intent: troubleshooting
    urgency: high
    error_code: E07
    species: null
    standalone_question: What should I do if my display shows E07?
    question: My display shows E07. What should I do?
[retriever -> coordinator] inform
    queries: ["What should I do if my display shows E07?", "E07 error code", "display shows E07", "MV-200 display error E07", "error code E07"]
    chunks: ["9. Warranty and Service (#10, 0.73)", "7. Error Codes (#7, 0.68)", "MycoVault MV-200 User & Service Manual (#0, 0.67)", "7. Error Codes (#8, 0.64)", "1. Product Overview (#1, 0.64)"]
[specialist -> coordinator] propose
    draft: E07 means the CO2 sensor drift is detected. To resolve this, follow these steps:  1. Move the unit to fresh air (or open a window). 2. Empty the chamber. 3. Hold FAN + LIGHT for 5 s. 4. Recalibration takes 8 minutes.
[verifier -> coordinator] inform
    supported: true
    unsupported_claims: []
  

### Part C — answers (from REPORT.md §1–§2 + deliverable Q1)

**Roles:** Triage classifies intent / urgency / error code and rewrites a `standalone_question`; Retriever expands queries and filters by τ; Specialist cites sections only from context; Verifier fact-checks; Escalation may open tickets.

**MAS vs pipeline?** Each agent has its own goal and prompt; they communicate via structured blackboard messages; the coordinator’s retry / escalation / short-circuit policy can change control flow (`reject`, `NOT_IN_MANUAL`) — not a fixed function call chain.

**History:** After E07, “and how long does that take?” is rewritten (e.g. to “How long does recalibration take?”) and answered with **8 minutes** from Section 7.

## Part D — Tools and escalation (D1)

Three cases: no ticket, medium ticket, high ticket. Guardrails validate tool calls against Section 9.

In [9]:
d1_cases = [
    ("No ticket", "My display shows E07. What should I do?"),
    ("Medium-priority ticket", "I got E19 this morning and again now."),
    ("High-priority ticket", "There is water under the cabinet and a burning smell!"),
]

coord_d = Coordinator(KnowledgeBase())
for title, q in d1_cases:
    print("\n" + "=" * 60)
    print(f"=== {title} ===")
    print("Q:", q)
    reply = coord_d.answer(q)
    print(coord_d.trace())
    print("\nReply:", reply)
    print("Ticket:", coord_d.last.get("ticket"))


=== No ticket ===
Q: My display shows E07. What should I do?
[triage -> coordinator] inform
    intent: troubleshooting
    urgency: high
    error_code: E07
    species: null
    standalone_question: What should I do if my display shows E07?
    question: My display shows E07. What should I do?
[retriever -> coordinator] inform
    queries: ["What should I do if my display shows E07?", "E07 error code", "display shows E07", "MV-200 display error E07", "error code E07"]
    chunks: ["9. Warranty and Service (#10, 0.73)", "7. Error Codes (#7, 0.68)", "MycoVault MV-200 User & Service Manual (#0, 0.67)", "7. Error Codes (#8, 0.64)", "1. Product Overview (#1, 0.64)"]
[specialist -> coordinator] propose
    draft: E07 means the CO2 sensor drift is detected. To resolve this, move the unit to fresh air (or open a window), empty the chamber, then hold FAN + LIGHT for 5 s. Recalibration takes 8 minutes.
[verifier -> coordinator] inform
    supported: true
    unsupported_claims: []
    feedbac

### Part D — answers (from REPORT.md §4 + §7.2)

| Case | Expected |
|------|----------|
| E07 once | no ticket (`lookup_error_code` only) |
| E19 twice | medium ticket |
| Water + burning smell | high ticket + unplug |

On the full E1 run: escalation **precision 1.00**, **recall 0.50**, **F1 0.67**. Tickets opened for Q11/Q13; gold tickets for Q12 (E23 > 10 min) and Q14 (E07 back after recalibration) were missed. Guardrails **block** unjustified tickets; they do **not** force missing ones. Full saved traces: `results/d1_traces.md`.

## Part E — Evaluation results (loaded, not re-run)

Displays tables already written by `evaluate.py` / `--ablation verifier`.

In [10]:
results = ROOT / "results"
for name in ("eval_table.md", "ablation_verifier.md"):
    path = results / name
    print(f"\n##### {name} #####\n")
    if path.exists():
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print(f"Missing {path}. Run: python evaluate.py  and  python evaluate.py --ablation verifier")


##### eval_table.md #####



# E1 evaluation (llama3.1:8b, 20 questions)

Judge model: `llama3.1:8b`. Tau = 0.5, chunk size = 900. Escalation metrics are n/a for baseline and RAG because they have no ticket tool. Hit@k uses k = 4 for RAG and up to 5 chunks above tau for the multi-agent system.

| Metric | Baseline | Single RAG (B3) | Multi-agent |
|---|---|---|---|
| Fact accuracy (string match) | 18 % | 73 % | 83 % |
| Fact accuracy (LLM judge) | 23 % | 59 % | 60 % |
| Refusal correctness (Q18, Q20) | 0 % | 100 % | 100 % |
| False refusals on in-manual questions | 0 | 2 | 1 |
| Escalation precision | n/a | n/a | 1.00 |
| Escalation recall | 0.00 | 0.00 | 0.50 |
| Escalation F1 | n/a | n/a | 0.67 |
| Ticket priority correct | n/a | n/a | 2/2 |
| Retrieval hit@k | n/a | 94 % | 100 % |
| Mean latency (s / question) | 3.13 | 0.66 | 4.00 |
| Mean LLM calls / question | 1.00 | 1.00 | 4.95 |

## Per question

| Q | baseline facts | rag facts | mas facts | MAS ticket | MAS retrieved gold |
|---|---|---|---|---|---|
| 1 | 0/4 | 4/4 | 4/4 | - | yes |
| 2 | 0/1 | 0/1 | 1/1 | - | yes |
| 3 | 0/5 | 3/5 | 3/5 | - | yes |
| 4 | 0/2 | 2/2 | 2/2 | - | yes |
| 5 | 2/3 | 3/3 | 3/3 | - | yes |
| 6 | 1/1 | 0/1 | 1/1 | - | yes |
| 7 | 0/1 | 1/1 | 1/1 | - | yes |
| 8 | 0/3 | 3/3 | 1/3 | - | yes |
| 9 | 0/5 | 5/5 | 5/5 | - | yes |
| 10 | 0/5 | 5/5 | 5/5 | - | yes |
| 11 | 0/2 | 2/2 | 1/2 | medium | yes |
| 12 | 0/2 | 1/2 | 1/2 | - | yes |
| 13 | 1/2 | 0/2 | 2/2 | high | yes |
| 14 | 0/1 | 1/1 | 1/1 | - | yes |
| 15 | 1/2 | 2/2 | 2/2 | - | yes |
| 16 | 1/2 | 2/2 | 2/2 | - | yes |
| 17 | 0/1 | 1/1 | 1/1 | - | yes |
| 18 | answered | refused | refused | - | n/a |
| 19 | 0/1 | 0/1 | 0/1 | - | yes |
| 20 | answered | refused | refused | - | n/a |



##### ablation_verifier.md #####



# E2 ablation: Verifier on vs off

Same chat model (`llama3.1:8b`), same KB, prompts and judge (`llama3.1:8b`). Only whether the Verifier + revision loop runs changes — no second model is loaded (avoids VRAM swapping).

Verifier ON source: `eval_llama3.1-8b.json`.

| Metric | Verifier ON (E1 MAS) | Verifier OFF |
|---|---|---|
| Fact accuracy (string match) | 83 % | 82 % |
| Fact accuracy (LLM judge) | 60 % | 63 % |
| Refusal correctness (Q18, Q20) | 100 % | 100 % |
| False refusals on in-manual questions | 1 | 0 |
| Escalation precision | 1.00 | 1.00 |
| Escalation recall | 0.50 | 0.50 |
| Escalation F1 | 0.67 | 0.67 |
| Ticket priority correct | 2/2 | 2/2 |
| Retrieval hit@k | 100 % | 100 % |
| Mean latency (s / question) | 4.00 | 2.81 |
| Mean LLM calls / question | 4.95 | 3.90 |


### Part E — answers (from REPORT.md §5–§6 + deliverable Q2–Q3)

**E1 headline:** baseline facts ~**18%** → RAG ~**73%** → MAS ~**83%**. Refusal on Q18/Q20: **0% → 100%**. MAS is slower (~4 s, ~5 LLM calls) than single RAG (~0.7 s, 1 call) but adds escalation and better hit@k.

**E2 Verifier on vs off (same `llama3.1:8b`, no VRAM swap):** OFF saves ~**1.2 s** and ~**1 call**/question with almost the same fact scores. Escalation metrics unchanged (Section 9 tools/guardrails).

**Verifier value / cost?** Meant to catch unsupported claims and missing safety warnings; on this 20-Q set mean facts barely change when off. Cost ≈ +1 call and +1.2 s/question.

**Same-model Verifier weakness?** Shared blind spots with the Specialist. Mitigate with a different/stronger judge, rule-based number checks against chunks, or require cited spans.

## Closing reflections (from REPORT.md §7–§8)

### Three failure cases
1. **Bleach retrieval** — dense search misses Maintenance first.  
2. **Escalation recall** — Q12/Q14 gold tickets missed (precision high, recall 0.5).  
3. **Partial facts / false refusal** — grounding can omit key facts or over-refuse when context is thin.

### Prompt injection (deliverable Q4)
Malicious text inside a retrieved chunk is inserted into Specialist/Verifier CONTEXT and can try to override “answer only from the manual”. Defend by treating chunks as untrusted data, stripping instruction-like lines, citing only spans, and keeping ticket tools behind the Section 9 guardrail.

Full write-up: **`REPORT.md`**. Reproduce scripts: **`README.md`**.